# Designing two comb lines at once with `InverseDesignMulti`

A coherent photonic tensor core has to hold **one** mask across every comb line *and* every
input port: the same physical structure is interrogated at 1545, 1550 and 1555 nm by four
lanes, and the weights at all of those wavelengths have to come out right simultaneously.
The freeform region is therefore a single unknown that many FDTD jobs share.

This notebook builds that problem at a fraction of the cost of the parent design — two comb
lines, two ports, four simulations per optimizer step, a tenth of the design pixels — using
`tidy3d.plugins.invdes.InverseDesignMulti`. It stops before any solve: it assembles the jobs,
checks that each one is uploadable, and estimates the FlexCredit cost of one epoch so the
budget is known before the first task is submitted.

## Why the comb is a set of jobs, not one job

The obvious organisation is one simulation with three frequencies and an objective summed
over them. Tidy3D's inverse-design plugin rejects that: `InverseDesign` validates the metric
frequency (`InverseDesign._validate_metric_f`) and raises when an objective carries more than
one, because the adjoint field of a multi-frequency objective is not a single solve. A design
simulation without a source is rejected at upload for the same class of reason.

So the comb becomes one simulation per `(comb line, input port)` pair — here 2 x 2 = 4 —
each with its own mode source at its own line frequency. `InverseDesignMulti` ties them
together: one `TopologyDesignRegion`, one density, one gradient step that sees every job, so
the mask cannot satisfy 1550 nm at the expense of 1555 nm.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tidy3d as td
import tidy3d.plugins.invdes as tdi
import tidy3d.web as web
import autograd.numpy as anp
from tidy3d.plugins.expressions.metrics import ModeAmp

print("tidy3d", td.__version__)

WAVELENGTHS_UM = (1.550, 1.555)                    # two comb lines
FREQS = td.C_0 / np.array(WAVELENGTHS_UM)          # Hz
K, N = len(WAVELENGTHS_UM), 2                      # comb lines x input ports

PORT_PITCH = 2.0                                   # um, half the 4.0 um lane pitch
DESIGN_LX, DESIGN_LY = 6.0, 6.0                    # freeform region, um
PIXEL = 0.040                                      # density pixel, um
FILTER_RADIUS = 0.140                              # minimum feature, um

In [ ]:
T_SIN, T_GAP, T_TFLN, WG_WIDTH = 0.300, 0.050, 0.300, 0.800   # um
N_SIO2, N_SIN = 1.444, 1.980
N_TFLN_O, N_TFLN_E = 2.211, 2.138                  # X-cut: ordinary in-plane, extraordinary out
STUB_LEN, PORT_PAD = 1.5, 0.4

sio2 = td.Medium(permittivity=N_SIO2**2, name="SiO2")
sin = td.Medium(permittivity=N_SIN**2, name="SiN")
tfln = td.AnisotropicMedium(
    xx=td.Medium(permittivity=N_TFLN_O**2),
    yy=td.Medium(permittivity=N_TFLN_O**2),
    zz=td.Medium(permittivity=N_TFLN_E**2),
    name="TFLN",
)


def port_y(index):
    """Centre of lane `index`, with the ports centred on y = 0."""
    return (index - (N - 1) / 2) * PORT_PITCH


def design_simulation(line, input_index):
    """One comb line, one input port: its own source, a monitor on every output.

    The SiN access stubs from every port run up to the design-region edge, so the region
    the optimizer sees is the one the fabricated core actually connects to. The TFLN slab
    overshoots the region by 3 um on each side and the domain by more, so no slab facet
    sits on a monitor plane.
    """
    freq = float(FREQS[line])
    x_source = -DESIGN_LX / 2 - 1.0
    x_monitor = DESIGN_LX / 2 + 1.0
    stub_center = (DESIGN_LX + STUB_LEN) / 2.0

    structures = []
    for index in range(N):
        for sign, side in ((-1.0, "in"), (1.0, "out")):
            structures.append(
                td.Structure(
                    geometry=td.Box(
                        center=(sign * stub_center, port_y(index), 0.0),
                        size=(STUB_LEN, WG_WIDTH, T_SIN),
                    ),
                    medium=sin,
                    name=f"SiN_stub_{index}_{side}",
                )
            )
    structures.append(
        td.Structure(
            geometry=td.Box(
                center=(0.0, 0.0, T_SIN / 2 + T_GAP + T_TFLN / 2),
                size=(DESIGN_LX + 6.0, DESIGN_LY + 6.0, T_TFLN),
            ),
            medium=tfln,
            name="TFLN",
        )
    )

    return td.Simulation(
        size=(DESIGN_LX + 2 * STUB_LEN + 3.0, DESIGN_LY + 8.0, 2.8),
        run_time=500.0 / freq,
        grid_spec=td.GridSpec.auto(min_steps_per_wvl=10, wavelength=1.55),
        structures=structures,
        sources=[
            td.ModeSource(
                center=(x_source, port_y(input_index), 0.0),
                size=(0.0, 2.5, T_SIN + 1.5),
                source_time=td.GaussianPulse(freq0=freq, fwidth=0.15 * freq),
                direction="+",
                mode_spec=td.ModeSpec(num_modes=1),
                name="input",
            )
        ],
        monitors=[
            td.ModeMonitor(
                center=(x_monitor, port_y(out), 0.0),
                size=(0.0, 2.0, T_SIN + 1.5),
                freqs=[freq],
                mode_spec=td.ModeSpec(num_modes=1),
                name=f"out_{out}",
            )
            for out in range(N)
        ],
        medium=sio2,
        boundary_spec=td.BoundarySpec.all_sides(boundary=td.PML()),
        plot_length_units="um",
    )

## What the gradient sees

Verification fits a pilot gain and a per-lane phase before it scores a kernel, so the design
objective must be blind to those two quantities. A per-element error against an absolute
`alpha * W * exp(i phi_ref)` spends gradient on a constant the receiver calibrates away, and
a mask that transmits nothing can still look self-consistent.

Each `(line, input)` job therefore contributes a normalised overlap

    |<g, t>|^2 / (||g||^2 ||t||^2),   g = W[k][:, i] * exp(i phi_ref[k])

which is invariant to a global phase and to a real gain — and, because it divides out
`||t||`, also blind to how much light a port receives. On its own a dead port still scores
whatever `|<g, t>|^2` its stray field gives: about 0.5 for random relative phase with two
outputs, and up to 1.0 with the right phase pattern.

Gating each output separately does not work either. One forward solve with the design region
filled solid delivers `|t| = 0.7445` to one output and `0.2571` to the other for the same
input, and a field with the right shape scaled to clear 0.8 of the larger of those needs
amplitude 1.31 on the smaller — more than a unit input carries. So the gate is on the total
transmitted norm: every job must carry `THROUGHPUT_FLOOR * STRAIGHT_NORM[line]` — 0.630 at
1550 nm and 0.630 at 1555 nm, from references of 0.7877 and 0.7872 — or it is charged
`THROUGHPUT_WEIGHT * shortfall^2 / STRAIGHT_NORM[line]^2`. Each comb line is calibrated on
its own straight-through run rather than sharing one scalar, so a line that couples worse
does not inherit another line's budget. With `FLOOR = 0.8` a port carrying a quarter of the
calibrated power scores -2.03 per job, where a half-amplitude floor on
`ALPHA_SCALE * ||target||` let it keep a full 1.0. The sum stays differentiable — the gate
is a rectifier, not a branch — and its gradient w.r.t. the shared density is the adjoint of
all four jobs at once, which is the whole point of `InverseDesignMulti`.

`design_objective(tasks, task_names)` returns that function, and `build_design` below calls
it with the task names of the design it just assembled, so the two are always created
together. The returned function also carries `.jobs`, the per-job amplitude ratio and
overlap of the last forward evaluation, which the schedule writes into its history.

In [ ]:
rng = np.random.default_rng(7)
WEIGHTS = rng.uniform(-0.5, 0.5, size=(K, N, N))       # target (lines, out, in)
PHI_REF = np.array([0.0, 0.35])[:K]
# |t| per output for a lossless straight-through path, measured once per comb line with the
# design region filled solid (input 0): 1550 nm on the first row, 1555 nm on the second.
STRAIGHT_AMP = np.array([[0.7445, 0.2571],
                         [0.7437, 0.2579]])
STRAIGHT_NORM = np.linalg.norm(STRAIGHT_AMP, axis=1)   # one reference amplitude per line
THROUGHPUT_FLOOR, THROUGHPUT_WEIGHT = 0.8, 10.0

targets = WEIGHTS * np.exp(1j * PHI_REF)[:, None, None]


def design_objective(tasks, task_names):
    """Phase-invariant overlap minus a per-line throughput gate, over every job."""
    amps = {
        (line, index, out): ModeAmp(
            monitor_name=f"out_{out}",
            freqs=float(FREQS[line]),
            direction="+",
            mode_index=0,
        )
        for line, index in tasks
        for out in range(N)
    }
    jobs = {}                                    # filled on every forward evaluation

    def post_process_fn(batch):
        total = anp.zeros(())
        for (line, index), task_name in zip(tasks, task_names):
            data = batch[task_name]
            measured = anp.stack(
                [amps[(line, index, out)].evaluate(data) for out in range(N)]
            )
            target = targets[line][:, index]
            target_norm2 = float(np.sum(np.abs(target) ** 2))
            measured_norm2 = anp.sum(anp.abs(measured) ** 2)
            inner = anp.sum(anp.conj(target) * measured)

            # shape: invariant to a global phase and to a real gain
            total = total + anp.abs(inner) ** 2 / (target_norm2 * measured_norm2 + 1e-30)

            # amplitude: the overlap alone cannot tell a quiet port from a lossless one,
            # so the job must also carry FLOOR of its own line's straight-through amplitude
            reference = float(STRAIGHT_NORM[line])
            shortfall = anp.maximum(
                0.0, THROUGHPUT_FLOOR * reference - anp.sqrt(measured_norm2 + 1e-30)
            )
            total = total - THROUGHPUT_WEIGHT * shortfall**2 / reference**2

            if isinstance(measured, np.ndarray):   # forward pass, not an autograd trace
                jobs[f"line{line}_in{index}"] = {
                    "amplitude_ratio": float(
                        np.sqrt(float(measured_norm2)) / reference
                    ),
                    "overlap": float(
                        np.abs(complex(inner)) ** 2
                        / (target_norm2 * float(measured_norm2) + 1e-30)
                    ),
                }
        return total

    post_process_fn.jobs = jobs
    return post_process_fn


print(f"targets {WEIGHTS.shape}, one 2x2 matrix per comb line")
print(f"straight-through references: {np.round(STRAIGHT_NORM, 4).tolist()}")

## The multi-job

`TopologyDesignRegion` is the only unknown: a 6 x 6 um patch of SiN that either exists
(`n = 1.980`) or does not (`n = 1.444` on oxide), discretised at 40 nm — about 3.5 pixels per
140 nm minimum feature. Two transformations shape it:

- `FilterProject(radius=0.140, beta=...)` smooths the density on the minimum-feature scale
  and then projects it towards 0/1 with a steepness `beta` that grows every epoch.
- `ErosionDilationPenalty(length_scale=0.140, weight=1.0)` penalises masks whose eroded and
  dilated versions disagree, which is what keeps a map that is nominally binary from being
  a field of isolated pixels the fab cannot draw.

`output_monitor_names` tells the plugin which monitors the objective will read, per
simulation; the names are what the objective indexes by, not positions.

In [ ]:
TASKS = [(line, index) for line in range(K) for index in range(N)]
BETA_START, BETA_GROWTH = 2.0, 1.08


def beta_for(epoch):
    """Projection steepness at optimizer epoch `epoch` (1-based)."""
    return BETA_START * BETA_GROWTH**epoch


# The epochs already on disk ran the 1.08 schedule above. Past the checkpoint the map is
# mostly binary and the projection gradient in a 240 nm gap is nearly zero, so the
# continuation holds beta low first and only then sharpens.
CHECKPOINT_STEP = 10                     # epochs already run and checkpointed
HOLD_BETA, HOLD_EPOCHS, HOLD_LR = 3.2, 10, 0.03
ANNEAL_STEPS, ANNEAL_STEP, BETA_MAX = 5, 0.4, 5.0
EPOCHS = CHECKPOINT_STEP + HOLD_EPOCHS + ANNEAL_STEPS


def beta_schedule(step):
    """Projection steepness at continuation step `step` (1-based, absolute)."""
    if step <= CHECKPOINT_STEP:
        return beta_for(step)
    if step <= CHECKPOINT_STEP + HOLD_EPOCHS:
        return HOLD_BETA
    return min(HOLD_BETA + ANNEAL_STEP * (step - CHECKPOINT_STEP - HOLD_EPOCHS), BETA_MAX)


def lr_schedule(step):
    """Adam step size: gentler while beta is held, back to 0.05 for the anneal."""
    if step <= CHECKPOINT_STEP:
        return 0.05                      # the epochs already run and checkpointed
    return HOLD_LR if step <= CHECKPOINT_STEP + HOLD_EPOCHS else 0.05


def build_design(beta):
    """The one unknown every job shares, and the objective that grades it.

    Both are assembled here, at the same `beta`, so the objective can never be left
    pointing at a structure the simulation no longer uses. The schedule rebuilds the
    pair every epoch; nothing else about the problem changes.
    """
    design = tdi.InverseDesignMulti(
        design_region=tdi.TopologyDesignRegion(
            size=(DESIGN_LX, DESIGN_LY, T_SIN),
            center=(0.0, 0.0, 0.0),
            eps_bounds=(N_SIO2**2, N_SIN**2),
            pixel_size=PIXEL,
            transformations=(
                tdi.FilterProject(radius=FILTER_RADIUS, beta=beta),
            ),
            penalties=(
                tdi.ErosionDilationPenalty(length_scale=FILTER_RADIUS, weight=1.0),
            ),
        ),
        simulations=tuple(
            design_simulation(line, index) for line, index in TASKS
        ),
        task_name="two_line_core",
        output_monitor_names=tuple(
            tuple(f"out_{out}" for out in range(N)) for _ in TASKS
        ),
    )
    return design, design_objective(TASKS, list(design.task_names))


design, objective_fn = build_design(beta_for(1))

for name in design.task_names:
    print(name)
print(f"{len(design.simulations)} simulations, one density, beta = {beta_for(1):.2f}")
print(f"objective built alongside it: {callable(objective_fn)}")

In [ ]:
simulation = design.simulations[0]
fig, ax = plt.subplots(figsize=(7, 5))
simulation.plot(z=0.0, ax=ax)
ax.add_patch(
    plt.Rectangle(
        (-DESIGN_LX / 2, -DESIGN_LY / 2), DESIGN_LX, DESIGN_LY,
        fill=False, ls="--", lw=1.5, ec="k", label="design region",
    )
)
ax.set_xlim(-7.5, 7.5)
ax.set_ylim(-7.5, 7.5)
ax.set_title("line 0, input 0: stubs, freeform region, source at left, monitors at right")
ax.legend(loc="lower right")
plt.show()

## Starting point

The optimizer starts from a mid-grey density, which the projection turns into a mottled
pattern that rarely has material in front of a port. Every port therefore gets a solid 0.4 um
landing seeded into the initial map, well inside the 0.6 um a fabricated core is allowed to
run past its stub tip: the adjacency is a starting condition, not a rule that gets relaxed
later. The objective can still remove it, but it starts connected.

In [ ]:
columns = int(round(DESIGN_LX / PIXEL))
rows = int(round(DESIGN_LY / PIXEL))
density0 = np.full((columns, rows), 0.5)

pad = max(int(round(PORT_PAD / PIXEL)), 1)
half_width = max(int(round(0.5 * WG_WIDTH / PIXEL)), 1)
for index in range(N):
    row = int(round((port_y(index) + DESIGN_LY / 2) / PIXEL))
    low, high = max(row - half_width, 0), min(row + half_width, rows)
    density0[:pad, low:high] = 1.0
    density0[columns - pad :, low:high] = 1.0

print(f"density {density0.shape} at {PIXEL * 1e3:.0f} nm")
print(f"seeded landings: {int((density0 == 1.0).sum())} pixels per 0.4 um pad")
print(f"freeform feature: {FILTER_RADIUS * 1e3:.0f} nm = {FILTER_RADIUS / PIXEL:.1f} pixels")

## Upload validation

`Optimizer.validate_pre_upload` dereferences `design.simulation`, a single-simulation
attribute `InverseDesignMulti` does not have — and it stops at the first failure. Validate
each simulation directly instead, naming the one that fails, so a bad job is found here
rather than after three good ones have been uploaded.

In [ ]:
for name, simulation in zip(design.task_names, design.simulations):
    simulation.validate_pre_upload()
    print(f"{name}: uploadable")

## What one epoch costs

The cost is set by the number of jobs per epoch and by the FDTD volume each one fills. The
full-size design in the parent project runs 12 jobs (3 lines x 4 ports) in a
24 x 28 x 2.8 um box and bills about **0.32 FlexCredits per job**, around **3.9 per epoch**.
This version runs 4 jobs in a 12 x 14 x 2.8 um box, a quarter of the volume.
`Job.estimate_cost` returns **0.027 per job**, forward-only, and one measured epoch billed
**0.20**: 0.025 for each forward solve plus 0.025 for its adjoint. That is **0.20 per
epoch** and roughly **2.4 FlexCredits** for the 12-epoch schedule, which is what the factor
below reproduces.

`Job.estimate_cost` reports the forward upload only. On the full-size design it returned
0.074 for a job that actually billed 0.32; here it returned 0.027 against a measured 0.025,
so on this problem the estimate is close and it is the adjoint solve that roughly doubles
the bill. The cell deletes every probe task once its estimate has been read, so running it
repeatedly leaves no dead jobs behind.

In [ ]:
FORWARD_TO_FULL = 1.85  # measured: 0.20 billed per epoch for 0.108 forward-only

per_job, probes = {}, []
try:
    for name, simulation in zip(design.task_names, design.simulations):
        job = web.Job(simulation=simulation, task_name=f"{name}_cost_probe")
        job.upload()
        probes.append(job)
        per_job[name] = job.estimate_cost()
        print(f"{name}: {per_job[name]:.3f} FlexCredits (estimate, forward only)")
finally:
    # estimate_cost only needs the uploaded task; leaving it behind would accumulate
    # probe jobs on the portal on every run, so delete each one once it is read.
    deleted, failed = 0, []
    for job in probes:
        try:
            web.delete(job.task_id)
            deleted += 1
        except Exception as exc:
            failed.append((job.task_id, exc))
    print(f"deleted {deleted} of {len(probes)} probe tasks")
    for task_id, exc in failed:
        print(f"  {task_id}: {type(exc).__name__}: {exc}")

forward_only = sum(per_job.values())
print()
print(f"{len(TASKS)} jobs, forward-only estimate:      {forward_only:.2f}")
print(f"planning cost of one epoch:             {forward_only * FORWARD_TO_FULL:.2f}")
print(f"planning cost of {EPOCHS} epochs:            {forward_only * FORWARD_TO_FULL * EPOCHS:.1f}")

## Checking the map between epochs

The anneal can close a gap that the hold phase has just opened, and a sweep that finds out only
in verification has already paid for the remaining steps. This check reads the projected map the
optimizer is working on — no simulation, no credits — and reports two things: how many separate
solid pieces the map has and whether all four port landings sit in the main one, and whether any
feature is thinner or any gap narrower than the 140 nm rule. The loop below runs it after every
epoch, and stops the anneal as soon as a port comes loose.

In [ ]:
import scipy.ndimage as ndi

N_CELLS = int(round(DESIGN_LX / PIXEL))            # 150 x 150 pixels
COORDS = (np.arange(N_CELLS) - (N_CELLS - 1) / 2) * PIXEL
UPSAMPLE = 4                                       # 40 nm pixels -> 10 nm sampling


def _disk(radius_px):
    r = int(np.ceil(radius_px))
    yy, xx = np.mgrid[-r : r + 1, -r : r + 1]
    return (xx**2 + yy**2) <= radius_px**2


def _specks(mask, min_px=4):
    """Drop isolated one- and two-pixel specks: a round opening erodes the corners of any
    axis-aligned rectangle, legal or not, and those specks are not rule violations."""
    labels, count = ndi.label(mask)
    if not count:
        return mask
    keep = np.concatenate(([False], np.bincount(labels.ravel())[1:] >= min_px))
    return mask & keep[labels]


def geometry_report(projected):
    """Connectivity of the four landings and the 140 nm rule, from one projected map.

    `projected` is the filter+projection output the optimizer differentiates, so this
    describes the map the next epoch starts from rather than the raw density. Pure
    numpy/scipy: no solve, so it can run after every epoch for free.
    """
    # the projection of a 0.5 density is 0.5 plus FFT roundoff, so the threshold
    # needs a tolerance or a uniform map reads as thousands of specks
    solid = np.asarray(projected).squeeze() > 0.5 + 1e-9
    labels, count = ndi.label(solid, structure=np.ones((3, 3), bool))
    sizes = np.bincount(labels.ravel())[1:]
    main = int(1 + np.argmax(sizes)) if count else 0

    pieces = {}
    for index in range(N):
        rows = np.nonzero(np.abs(COORDS - port_y(index)) <= WG_WIDTH / 2)[0]
        for sign, side in ((-1.0, "in"), (1.0, "out")):
            cols = np.nonzero(
                np.abs(COORDS - sign * (DESIGN_LX / 2 - PORT_PAD / 2)) <= PORT_PAD / 2
            )[0]
            pieces[f"line{index}_{side}"] = int(labels[np.ix_(cols, rows)].max())

    # a landing counts as routed only when all four sit in the same solid piece: an island of
    # its own is not a route, however large it happens to be
    shared = list(pieces.values())
    group = {piece: shared.count(piece) for piece in set(shared)}
    ports = {
        name: bool(piece and group[piece] == len(shared)) for name, piece in pieces.items()
    }

    # a 70 nm disk at 10 nm sampling is the 140 nm rule, applied to the same solid mask
    disk = _disk(FILTER_RADIUS / 2 / (PIXEL / UPSAMPLE))
    fine = np.repeat(np.repeat(solid, UPSAMPLE, 0), UPSAMPLE, 1)
    thin = _specks(solid & ~ndi.binary_opening(fine, structure=disk)[::UPSAMPLE, ::UPSAMPLE])
    narrow = _specks(
        ~solid & ndi.binary_closing(fine, structure=disk)[::UPSAMPLE, ::UPSAMPLE]
    )

    return {
        "solid": float(solid.mean()),
        "components": int(count),
        "largest": [int(s) for s in np.sort(sizes)[::-1][:4]],
        "ports": ports,
        "pieces": pieces,
        "connected": all(ports.values()),
        "thin_px": int(thin.sum()),
        "narrow_px": int(narrow.sum()),
    }


def format_report(report):
    """One line per epoch, so the report stays readable in a run log."""
    empty = [name for name, piece in report["pieces"].items() if not piece]
    if report["connected"]:
        where = "all four landings in one piece"
    elif empty:
        where = "split, nothing at " + ", ".join(empty)
    else:
        split = len({piece for piece in report["pieces"].values() if piece})
        where = f"landings split across {split} pieces"
    return (
        f"map: {report['components']} piece(s) {report['largest']}, "
        f"solid {report['solid']:.2f}; {where}; "
        f"140 nm: {report['thin_px']} px thin, {report['narrow_px']} px narrow"
    )


# the seeded start: four isolated pads, so the loop below has a baseline to beat
seeded_design, _ = build_design(beta_schedule(CHECKPOINT_STEP + 1))
seeded_map = seeded_design.design_region.evaluate_transformation(
    seeded_design.design_region.transformations[0], density0[:, :, None]
)
print(format_report(geometry_report(seeded_map)))

## The schedule

`beta` is one value per epoch, and the design region is rebuilt at that beta before each
step. Past the checkpoint the schedule holds `beta` at 3.2 with a gentler learning rate for
ten steps, so the map stays plastic enough to close the port-1 gaps, and only then anneals
0.4 per step to 5.0. The
`design` and `objective` both come from `build_design`, so this schedule, the object
described above and the objective that grades it cannot drift apart; it resumes the
previous step's density instead of starting over:

```python
import json
from pathlib import Path

result, start = None, 1
out = Path("two_line_checkpoints")
out.mkdir(exist_ok=True)
history = {"beta": [], "objective": [], "penalty": [], "jobs": {}, "geometry": {}}

done = sorted(out.glob("epoch_*.npz"))
if done:                                      # pick up an interrupted sweep
    start = int(done[-1].stem.split("_")[1]) + 1
    params0 = np.load(done[-1])["density"][:, :, None].astype(float)
    history = json.loads((out / "history.json").read_text())
    history.setdefault("jobs", {})
    history.setdefault("geometry", {})
    print(f"resuming at epoch {start} from {done[-1].name}")
else:
    params0 = density0[:, :, None].astype(float)

# a stopped anneal leaves this marker: the next run retries the step that broke a port at the
# hold beta, from the last map that still had all four landings, instead of sharpening into
# the same wall at the higher beta
stop_file = out / "anneal_stop.json"
stopped = json.loads(stop_file.read_text()) if stop_file.exists() else {}
if stopped:
    print(f"anneal stopped at epoch {stopped['epoch']}: retrying it at beta {HOLD_BETA}")

for epoch in range(start, EPOCHS + 1):
    retrying = stopped.get("epoch") == epoch
    beta = HOLD_BETA if retrying else beta_schedule(epoch)
    learning_rate = HOLD_LR if retrying else lr_schedule(epoch)
    design, objective = build_design(beta)   # both, as the runner does
    optimizer = tdi.AdamOptimizer(
        design=design, learning_rate=learning_rate, num_steps=1,
        results_cache_fname="two_line_history.hdf5",
    )
    if result is None:                        # seed from the resumed or initial density
        result = tdi.InverseDesignResult(
            design=design,
            opt_state=(optimizer.initial_state(params0),),
            params=(params0,),
        )
    result = optimizer.continue_run(
        result=result, num_steps=1, post_process_fn=objective,
    )

    # free: the same projected map the next step would start from
    report = geometry_report(
        design.design_region.evaluate_transformation(
            design.design_region.transformations[0], result.params[-1]
        )
    )
    if retrying and report["connected"]:      # the hold beta repaired the split
        stop_file.unlink(missing_ok=True)
        stopped = {}

    if epoch > CHECKPOINT_STEP + HOLD_EPOCHS and not report["connected"]:
        # the marker, not this map, is what the next run resumes from: a split map never
        # becomes a checkpoint, so nothing downstream starts from a broken state
        stop_file.write_text(
            json.dumps(
                {"epoch": epoch, "beta": beta, "retried_at_hold": retrying},
                indent=1,
            )
        )
        print("  " + format_report(report), flush=True)
        if retrying:
            print(
                "  still split at the hold beta: this step needs a seeded bridge, not a "
                "smaller step. Stopping.",
                flush=True,
            )
        else:
            print(
                f"  a port is disconnected after this anneal step, so it is not "
                f"checkpointed. The next run resumes at epoch {epoch} and retries it at "
                f"beta {HOLD_BETA}.",
                flush=True,
            )
        break

    # checkpoint before the next solve: a kill loses at most the step in flight
    np.savez_compressed(
        out / f"epoch_{epoch:02d}.npz",
        density=result.params[-1][:, :, 0],   # params holds every epoch; -1 is this step
        beta=beta,
    )
    history["beta"].append(beta)
    history["objective"].append(float(result.objective_fn_val[-1]))
    history["penalty"].append(float(result.penalty[-1]))
    # keyed by epoch, so a history seeded without jobs cannot go out of step
    history["jobs"][f"epoch_{epoch:02d}"] = dict(getattr(objective, "jobs", {}))
    history["geometry"][f"epoch_{epoch:02d}"] = report
    tmp = out / "history.json.tmp"
    tmp.write_text(json.dumps(history, indent=1))
    tmp.replace(out / "history.json")
    print(
        f"epoch {epoch:02d} beta={beta:.2f} lr={learning_rate:.2f} "
        f"objective={history['objective'][-1]:.6f} "
        f"penalty={history['penalty'][-1]:.6f}",
        flush=True,
    )
    for job_name, job in history["jobs"].get(f"epoch_{epoch:02d}", {}).items():
        print(
            f"  {job_name}: amplitude {job['amplitude_ratio']:.3f} "
            f"overlap {job['overlap']:.3f}",
            flush=True,
        )
    print("  " + format_report(report), flush=True)
```

One step per epoch means one forward and one adjoint solve per job per epoch — that is what
the cost above buys. Each step writes its density and the running history to
`two_line_checkpoints/` before the next solve starts, and the loop resumes from the newest
`epoch_*.npz`, so a killed run loses at most the step in flight and needs nothing from the
results cache. The per-epoch `SimulationData` is deliberately not kept: the optimizer holds
it only for the duration of the step and the portal does not serve those tasks afterwards.
The geometry report runs on the map the step just produced, before that step is written
anywhere: if a bridge pinches off during the anneal the step is not checkpointed, the run
stops, and `anneal_stop.json` records why. The next run therefore resumes from the last map
that still had all four landings and retries the failed step at the hold beta, rather than
sharpening straight back into the wall that split it.

In [ ]:
print(f"{'step':>5s} {'beta':>7s} {'lr':>5s}  phase")
for step in range(1, EPOCHS + 1):
    if step <= CHECKPOINT_STEP:
        phase = "done"
    elif step <= CHECKPOINT_STEP + HOLD_EPOCHS:
        phase = "hold"
    else:
        phase = "anneal"
    print(f"{step:5d} {beta_schedule(step):7.2f} {lr_schedule(step):5.2f}  {phase}")

## Limits

- Two ports at 2.0 um pitch is tighter than the 4.0 um lane pitch of the full core: the
  stubs and the 0.8 um waveguides leave 1.2 um between lanes, so this problem exercises the
  machinery, not the crosstalk budget of the real core.
- Two comb lines 5 nm apart share one mask here; the real design spans 10 nm, where the
  phase-slope tolerance is tighter and the adjoint has more to reconcile.
- Nothing is exported: the density would still have to be thresholded, contoured to the
  140 nm freeform rule and re-simulated as polygons before it could enter an S-matrix.
- The solve is not run in this notebook. The estimate above and the upload validation are
  everything that can be checked without spending credits.
- **Budget caps the schedule before accuracy does.** One measured epoch bills 0.20
  FlexCredits, and the continuation runs 15 more (ten at beta 3.2, five annealing to 5.0),
  so finishing needs about **3.0**. The balance is down to **0.21** after the straight-through
  calibrations, which covers a single epoch: top up by roughly **2.8** before resuming at
  epoch 11, or shorten the hold to six steps and finish for about **2.0**.

*Draft for the Flexcompute community library. The stack values are generic SiN/TFLN indices,
not foundry data.*